In [1]:
import sys
sys.path.append(r'./ETGEM/etgems_web/')
from ETGEM.etgems_web.script.io.json import *
from ETGEM.etgems_web.script.io.convert import *
from ETGEM.etgems_web.script.tcmodel.construct import *
import pandas as pd
import numpy as np
import cobra
import copy
import re
cobra_config = cobra.Configuration()
cobra_config.solver = "gurobi"

In [2]:
def reaction_create(rxn_id, name, lower_bound, upper_bound, metabolites, kcat=1000000, kcat_MW=1000000,notes={},annotation={},mw=3600000,g0=0): 
    return {"id": rxn_id, "name": name, "lower_bound": lower_bound, "upper_bound": upper_bound, \
    "metabolites": metabolites, "kcat": kcat, "kcat_MW": kcat_MW, "mw": mw, "notes": notes,     \
    "annotation": annotation, "g0": g0, "objective_coefficient": 0}

def newmet_create(bigmodel_met,compartment="c"):
    return {
      "id": bigmodel_met.id.replace("c0",compartment+"0"),
      "name": bigmodel_met.name,
      "compartment": compartment,
      "charge": bigmodel_met.charge,
      "formula": bigmodel_met.formula,
      "concentration_ub": -3.912023005,
      "concentration_lb": -14.50865774,
      "annotation": {},
    }

def process_metabolites(rxnlist, bigmodel):
    """
    处理反应列表中的代谢物信息
    :param rxnlist: 反应列表
    :param bigmodel: 提供代谢物详细信息的大模型
    :return: 存储代谢物信息的字典
    """
    metdict = {}
    # 遍历 rxnlist 中的每个反应
    for reaction in rxnlist:
        # 遍历当前反应中代谢物的 ID
        for metabolite_id in reaction['metabolites'].keys():
            if metabolite_id.endswith("c0"):
                # 从 bigmodel 中获取代谢物对象
                metabolite = bigmodel.metabolites.get_by_id(metabolite_id)
                # 构建包含代谢物详细信息的字典
                metabolite_info = {
                    'id': metabolite_id,
                    'name': metabolite.name,
                    'compartment': "c",
                    'charge': metabolite.charge,
                    'formula': metabolite.formula,
                    "concentration_ub": -3.912023005,
                    "concentration_lb": -14.50865774,
                    'annotation': metabolite.annotation
                }
            elif metabolite_id.endswith("p0"):
                rep_met = metabolite_id.replace("p0",'c0')
                # 从 bigmodel 中获取代谢物对象
                metabolite = bigmodel.metabolites.get_by_id(rep_met)
                # 构建包含代谢物详细信息的字典
                metabolite_info = {
                    'id': metabolite_id,
                    'name': metabolite.name,
                    'compartment': "p",
                    'charge': metabolite.charge,
                    'formula': metabolite.formula,
                    "concentration_ub": -3.912023005,
                    "concentration_lb": -14.50865774,
                    'annotation': metabolite.annotation
                }
            elif metabolite_id.endswith("e0"):
                rep_met = metabolite_id.replace("e0",'c0')
                # 从 bigmodel 中获取代谢物对象
                metabolite = bigmodel.metabolites.get_by_id(rep_met)
                # 构建包含代谢物详细信息的字典
                metabolite_info = {
                    'id': metabolite_id,
                    'name': metabolite.name,
                    'compartment': "e",
                    'charge': metabolite.charge,
                    'formula': metabolite.formula,
                    "concentration_ub": -3.912023005,
                    "concentration_lb": -14.50865774,
                    'annotation': metabolite.annotation
                }
            # 将代谢物信息存入 metdict 中
            metdict[metabolite_id] = metabolite_info
    return metdict

In [3]:
bigmodel = cobra.io.load_json_model("../ModelSeed_db4model_20260411.json")

Set parameter LicenseID to value 2845745


In [4]:
json_path = "./model/ETiZM564_MM_optimized.json"
model_json = json_load(json_path)
model_iZM564 = cobra.io.load_json_model("./model/iZM564_20260807_KEGG_met.json")
# model_json['reactions'].extend(bdo_rxnlist)
# model_json['metabolites'].extend(bdo_metlist)   
# etmodel_MM_23BDO_path = "./model/ETiZM564_MM_optimized_23BDO.json"
# with open(etmodel_MM_23BDO_path,"w")as wf:
#     json.dump(model_json,wf,indent=2)

# C2 Product
# 乙醇（Ethanol）

In [5]:
model = cobra.io.load_json_model('./model/ETiZM564_MM_optimized.json')
model.objective = "EX_etoh_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 65
O2_level = 0 # 1000
medium['EX_o2_e'] = O2_level
model.medium = medium
# model.reactions.get_by_id("ZM_BIOMASS").bounds = (0.1,0.1)
model.reactions.get_by_id("ATPM").bounds = (0,0)

pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 46

print(pfba_solution['EX_etoh_e'],pfba_solution['EX_etoh_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_etoh_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)

# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])

# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")
        

130.0 2.0 0.5111111111111111


In [6]:
model = read_json_model('./model/etiZM564_EG.json')
model.obj_name = "EX_etoh_e"
model.medium['EX_glc_D_e_reverse'] = 65
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 65)
O2_level = 1000 # 0|1000
model.medium['EX_o2_e_reverse'] = O2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, O2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 46

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         # percent = opt_etm.reaction[i].value/opt_etm.reaction["EX_glc_D_e_reverse"].value*100
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

# 乙二醇（ethylene glycol） Glucose

In [ ]:
rxn01305 = reaction_create('rxn01305',"1,2-Ethanediol:NAD+ oxidoreductase",0,1000,{"cpd00003_c0":1.0,"cpd00992_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd00229_c0":-1.0})
EX_12EDO_e = reaction_create('EX_12EDO_e','EX_12EDO_e exchange',0,1000,{'cpd00992_c0':-1.0})

In [ ]:
model_iZM564_EG = json_load(json_path)
rxnlist = [rxn01305,EX_12EDO_e]
for i in rxnlist:
    if i not in model_json['reactions']:
        model_iZM564_EG['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        model_iZM564_EG['metabolites'].append(metdict[newmet])


cpd00003_c0
cpd00004_c0
cpd00067_c0
cpd00229_c0


In [ ]:
with open('./model/etiZM564_EG.json',"w") as f:
    json.dump(model_iZM564_EG,f,indent=2)

In [ ]:
model = cobra.io.load_json_model('./model/etiZM564_EG.json')
model.objective = "EX_12EDO_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 65
o2_level = 0 # 0|1000
medium['EX_o2_e'] = o2_level
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 62

print(pfba_solution['EX_12EDO_e'],pfba_solution['EX_12EDO_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_12EDO_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)

# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

22.469135802469136 0.345679012345679 0.11906721536351166


In [ ]:
model = read_json_model('./model/etiZM564_EG.json')
model.obj_name = "EX_12EDO_e"
model.medium['EX_glc_D_e_reverse'] = 65
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 65)
o2_level = 0 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 62

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         # percent = opt_etm.reaction[i].value/opt_etm.reaction["EX_glc_D_e_reverse"].value*100
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

7.959372068666162 0.3456790123456893 0.11906721536351521


## 乙二醇（ethylene glycol） Xylose

In [ ]:
EX_xyl_D_e_reverse = reaction_create('EX_xyl_D_e_reverse','EX_xyl_D_e_reverse',0,1000,{"cpd00154_e0":1.0})
EXT_xyl_D_e = reaction_create('EXT_xyl_D_e','EX_xyl_D_e_reverse',0,1000,{"cpd00154_e0":-1.0,"cpd00154_c0":1.0})

rxn01041 =reaction_create('rxn01041','D-xylose:NAD+ 1-oxidoreductase',0,1000,{"cpd00003_c0":-1.0,"cpd00154_c0":-1.0,"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd01527_c0":1.0})
rxn01041_reverse =reaction_create('rxn01041_reverse','D-xylose:NAD+ 1-oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00154_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd01527_c0":-1.0})

rxn01751 =reaction_create('rxn01751','D-Xylonolactone lactonohydrolase',0,1000,{"cpd00001_c0":-1.0,"cpd01527_c0":-1.0,"cpd00067_c0":1.0,"cpd00391_c0":1.0})

rxn01753 =reaction_create('rxn01753','D-Xylonate hydro-lyase',0,1000,{"cpd00391_c0":-1.0,"cpd00001_c0":1.0,"cpd02386_c0":1.0})

rxn01306 =reaction_create('rxn01306','2-dehydro-3-deoxy-D-pentonate glycolaldehyde-lyase (pyruvate-forming)',0,1000,{"cpd02386_c0":-1.0,"cpd00020_c0":1.0,"cpd00229_c0":1.0})
rxn01306_reverse =reaction_create('rxn01306_reverse','2-dehydro-3-deoxy-D-pentonate glycolaldehyde-lyase (pyruvate-forming)',0,1000,{"cpd02386_c0":1.0,"cpd00020_c0":-1.0,"cpd00229_c0":-1.0})

rxn01305 = reaction_create('rxn01305','1,2-Ethanediol:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":-1.0,"cpd00992_c0":-1.0,"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd00229_c0":1.0})

rxn01305_reverse = reaction_create("rxn01305_reverse",'1,2-Ethanediol:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00992_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd00229_c0":-1.0})

EX_12EDO_e = reaction_create('EX_12EDO_e','EX_12EDO_e exchange',0,1000,{'cpd00992_c0':-1.0})

In [ ]:
model_iZM564_EG_xyl = json_load(json_path)
rxnlist = [EX_xyl_D_e_reverse,EXT_xyl_D_e,rxn01041,rxn01041_reverse,rxn01751,rxn01753,rxn01306,rxn01306_reverse,rxn01305_reverse,rxn01305,EX_12EDO_e]
for i in rxnlist:
    if i not in model_json['reactions']:
        model_iZM564_EG_xyl['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        model_iZM564_EG_xyl['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_EG_xyl.json',"w") as f:
    json.dump(model_iZM564_EG_xyl,f,indent=2)

cpd00154_e0
cpd00154_c0
cpd00003_c0
cpd00004_c0
cpd00067_c0
cpd00001_c0
cpd00020_c0
cpd00229_c0


In [ ]:
model = cobra.io.load_json_model('./model/etiZM564_EG_xyl.json')
model.objective = "EX_12EDO_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 0
medium['EX_xyl_D_e_reverse'] = 100
o2_level = 1000 # 0|1000
medium['EX_o2_e'] = o2_level
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 62

print(pfba_solution['EX_12EDO_e'],pfba_solution['EX_12EDO_e']/pfba_solution['EX_xyl_D_e_reverse'],pfba_solution['EX_12EDO_e']/pfba_solution['EX_xyl_D_e_reverse']*product_mass/150)
# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

100.0 1.0 0.41333333333333333


In [ ]:
model = read_json_model('./model/etiZM564_EG_xyl.json')
model.obj_name = "EX_12EDO_e"
model.medium['EX_glc_D_e_reverse'] = 0
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 0)
model.medium['EX_xyl_D_e_reverse'] = 100
model.get_reaction_by_id("EX_xyl_D_e_reverse").bounds = (0, 100)
o2_level = 0 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 62

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_xyl_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_xyl_D_e_reverse"].value*product_mass/150)
for i in model.reactions:
    if opt_etm.reaction[i].value > 1e-4:
        print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

100.0 1.0 0.41333333333333333
ACTNtex	(S)-Acetoin <=> (S)-Acetoin	(0.0, 1000.0)	50.000
ACTNtpp	(S)-Acetoin <=> (S)-Acetoin	(0.0, 1000.0)	50.000
EX_actn__S_e	(S)-Acetoin <=> 	(0.0, 1000.0)	50.000
EX_co2_e	CO2 <=> 	(0.0, 1000.0)	100.000
rxn00168	Pyruvate + H+ <=> CO2 + Acetaldehyde	(0.0, 1000.0)	50.000
rxn02115	ACTN <=> (S)-Acetoin	(0.0, 1000.0)	50.000
rxn08237	CO2 <=> CO2	(0.0, 1000.0)	100.000
rxn09385	Xylose <=> Xylose	(0.0, 1000.0)	100.000
rxn13394	Pyruvate + H+ + Acetaldehyde <=> CO2 + ACTN	(0.0, 1000.0)	50.000
rxn08238_reverse	CO2 <=> CO2	(0.0, 1000.0)	100.000
EX_xyl_D_e_reverse	 <=> Xylose	(0, 100)	100.000
rxn01041	NAD + Xylose <=> NADH + H+ + D-Xylonolactone	(0.0, 1000.0)	100.000
rxn01751	H2O + D-Xylonolactone <=> H+ + CPD0-1198	(0.0, 1000.0)	100.000
rxn01753	CPD0-1198 <=> H2O + 2-Dehydro-3-deoxy-D-xylonate	(0.0, 1000.0)	100.000
rxn01306	2-Dehydro-3-deoxy-D-xylonate <=> Pyruvate + Glycolaldehyde	(0.0, 1000.0)	100.000
rxn01305_reverse	NADH + H+ + Glycolaldehyde <=> NAD + 1,2-Ethane

# C3 Product
## 乳酸（lactate)

In [ ]:
model = cobra.io.load_json_model('./model/ETiZM564_MM_optimized.json')
model.objective = "EX_lac_D_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 65
O2_level = 1000 # 0|1000
medium['EX_o2_e'] = O2_level
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 89

print(pfba_solution['EX_lac_D_e'],pfba_solution['EX_lac_D_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_lac_D_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)
# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

130.0 2.0 0.9888888888888889


In [ ]:
model = read_json_model('./model/etiZM564_EG.json')
model.obj_name = "EX_lac_D_e"
model.medium['EX_glc_D_e_reverse'] = 65
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 65)
O2_level = 1000 # 0|1000
model.medium['EX_o2_e_reverse'] = O2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, O2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 89

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         # percent = opt_etm.reaction[i].value/opt_etm.reaction["EX_glc_D_e_reverse"].value*100
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

0.5788469697133012 2.0000000000039284 0.9888888888908312


# 1,3-PDO

In [ ]:
EXT_glyc_e = reaction_create('EXT_glyc_e','glycerol transporter',0,1000,{"cpd00100_e0":-1.0,"cpd00100_c0":1.0})
EX_glyc_e_reverse = reaction_create('EX_glyc_e_reverse','glycerol exchange',0,1000,{"cpd00100_e0":1.0})

rxn00768 = reaction_create('rxn00768','glycerol hydro-lyase (3-hydroxypropanal-forming)',0,1000,{"cpd00100_c0":-1.0,"cpd00001_c0":1.0,"cpd00714_c0":1.0})
# (1) Glycerol[0] => (1) H2O[0] + (1) 3-Hydroxypropanal[0]
rxn02235 = reaction_create('rxn02235','propane-1,3-diol:NAD+ 1-oxidoreductase',0,1000,{"cpd00003_c0":-1.0,"cpd01618_c0":-1.0,"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd00714_c0":1.0})
rxn02235_reverse = reaction_create('rxn02235_reverse','propane-1,3-diol:NAD+ 1-oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd01618_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd00714_c0":-1.0})
# (1) NAD[0] + (1) 1,3-Propanediol[0] <=> (1) NADH[0] + (1) H+[0] + (1) 3-Hydroxypropanal[0]
rxn00610 = reaction_create('rxn00610','R00841',0,1000,{"cpd00001_c0":-1.0,"cpd00080_c0":-1.0,"cpd00009_c0":1.0,"cpd00100_c0":1.0})
# (1) H2O[0] + (1) Glycerol-3-phosphate[0] <=> (1) Phosphate[0] + (1) Glycerol[0]
rxnPdup = reaction_create('rxnPdup','rxnPdup',0,1000,{"cpd00003_c0":-1.0,"cpd00010_c0":-1.0,"cpd00714_c0":-1.0,"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd03375_c0":1.0})
rxnPdup_reverse = reaction_create('rxnPdup_reverse','rxnPdup_reverse',0,1000,{"cpd00003_c0":1.0,"cpd00010_c0":1.0,"cpd00714_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd03375_c0":-1.0})
# nad +coa + 3-Hydroxypropanal = nadh + h + 3-Hydroxypropanoyl-CoA

EXT_pdo_e = reaction_create('EXT_pdo_e','1,3-Propanediol transporter',0,1000,{"cpd01618_c0":-1.0,"cpd01618_e0":1.0})
EX_pdo_e = reaction_create('EX_pdo_e','1,3-Propanediol exchange',0,1000,{"cpd01618_e0":-1.0})

In [ ]:
etiZM564_13PDO_glyc = json_load(json_path)
rxnlist = [EXT_glyc_e,EX_glyc_e_reverse,rxn00768,rxn02235,rxn02235_reverse,rxn00610,rxnPdup,rxnPdup_reverse,EXT_pdo_e,EX_pdo_e]
for i in rxnlist:
    if i not in model_json['reactions']:
        etiZM564_13PDO_glyc['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        etiZM564_13PDO_glyc['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_13PDO_glyc.json',"w") as f:
    json.dump(etiZM564_13PDO_glyc,f,indent=2)

cpd00100_e0
cpd00100_c0
cpd00001_c0
cpd00003_c0
cpd00004_c0
cpd00067_c0
cpd00080_c0
cpd00009_c0
cpd00010_c0
cpd03375_c0


In [ ]:
model = cobra.io.load_json_model('./model/etiZM564_13PDO_glyc.json')
model.objective = "EX_pdo_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 0
medium['EX_glyc_e_reverse'] = 100
o2_level = 0 # 0|1000
medium['EX_o2_e'] = o2_level
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 76

print(pfba_solution['EX_pdo_e'],pfba_solution['EX_pdo_e']/pfba_solution['EX_glyc_e_reverse'],pfba_solution['EX_pdo_e']/pfba_solution['EX_glyc_e_reverse']*product_mass/92)
# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

75.0 0.75 0.6195652173913043


In [ ]:
model = read_json_model('./model/etiZM564_13PDO_glyc.json')
model.obj_name = "EX_pdo_e"
model.medium['EX_glc_D_e_reverse'] = 0
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 0)
model.get_reaction_by_id("EX_glyc_e_reverse").bounds = (0, 100)
model.medium['EX_glyc_e_reverse'] = 100
o2_level = 1000 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 76

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glyc_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glyc_e_reverse"].value*product_mass/92)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

75.0000000000008 0.750000000000008 0.619565217391311


# 1,3_PDO Glucose

In [ ]:
# FBA test failed with no Enzyme and Thermodynamic constraint
model = cobra.io.load_json_model('./model/etiZM564_13PDO_glyc.json')
model.objective = "EX_pdo_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 40
medium['EX_glyc_e'] = 0
medium['EX_glyc_e_reverse'] = 0
o2_level = 0 # 0|1000
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
# model.reactions.get_by_id("rxn00612_reverse").bounds = (0,0)
# model.reactions.get_by_id("rxn05937_reverse").bounds = (0,0)
# model.reactions.get_by_id("rxn01100_reverse").bounds = (0,0)
# model.reactions.get_by_id("rxn00974").bounds = (0,0)

model.reactions.get_by_id("rxn42829").bounds = (0,0)
model.reactions.get_by_id("rxn15750_num1").bounds = (0,0)
model.reactions.get_by_id("rxn15750_num2").bounds = (0,0)

# rxn02260 改为rxn16146
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 76

print(pfba_solution['EX_pdo_e'],pfba_solution['EX_pdo_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_pdo_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)
# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

20.000000000000004 0.49999999999999983 0.21111111111111103


In [ ]:
model = read_json_model('./model/etiZM564_13PDO_glyc.json')
model.obj_name = "EX_pdo_e"
model.medium['EX_glc_D_e_reverse'] = 100
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 100)

model.get_reaction_by_id("EX_glyc_e_reverse").bounds = (0, 0)
model.medium['EX_glyc_e_reverse'] = 0
o2_level = 0 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)

product_mass = 76

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

40.850592103699896 0.5 0.2111111111111111


## 丙氨酸

In [ ]:
rxn00278 = reaction_create('rxn00278','L-Alanine:NAD+ oxidoreductase (deaminating)',0,1000,{"cpd00001_c0":-1.0,"cpd00003_c0":-1.0,"cpd00035_c0":-1.0,"cpd00004_c0":1.0,"cpd00013_c0":1.0,"cpd00020_c0":1.0,"cpd00067_c0":1.0})
rxn00278_reverse = reaction_create('rxn00278_reverse','L-Alanine:NAD+ oxidoreductase (deaminating)',0,1000,{"cpd00001_c0":1.0,"cpd00003_c0":1.0,"cpd00035_c0":1.0,"cpd00004_c0":-1.0,"cpd00013_c0":-1.0,"cpd00020_c0":-1.0,"cpd00067_c0":-1.0})
EXT_L_ala_e = reaction_create('EXT_L_ala_e','EXT_L_ala_e',0,1000,{"cpd00035_c0":-1.0,"cpd00035_e0":1.0})
EX_L_ala_e = reaction_create('EX_L_ala_e','EX_L_ala_e',0,1000,{"cpd00035_e0":-1.0})

In [ ]:
etiZM564_ala = json_load(json_path)
rxnlist = [rxn00278,rxn00278_reverse,EXT_L_ala_e,EX_L_ala_e]
for i in rxnlist:
    if i not in model_json['reactions']:
        etiZM564_ala['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        etiZM564_ala['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_ala.json',"w") as f:
    json.dump(etiZM564_ala,f,indent=2)

cpd00001_c0
cpd00003_c0
cpd00035_c0
cpd00004_c0
cpd00013_c0
cpd00020_c0
cpd00067_c0
cpd00035_e0


In [ ]:
model = cobra.io.load_json_model("./model/etiZM564_ala.json")
model.objective = "EX_L_ala_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 100
o2_level = 1000 # 0|1000
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
medium['EX_co2_e'] = 0
medium['EX_nh4_e'] = 1000
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 89

print(pfba_solution['EX_L_ala_e'],pfba_solution['EX_L_ala_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_L_ala_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)

# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

200.0 2.0 0.9888888888888889


In [ ]:
model = read_json_model('./model/etiZM564_ala.json')
model.obj_name = "EX_L_ala_e"
model.medium['EX_glc_D_e_reverse'] = 100
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 100)
o2_level = 0 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 89

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

200.00000000000006 2.0 0.9888888888888889


# C4 Product
## Isobutanol

In [ ]:
#kdcA
rxn24344 = reaction_create('rxn24344','RXN-7643.c',0,1000,{"cpd00067_c0":-1.0,"cpd00123_c0":-1.0,"cpd00011_c0":1.0,"cpd24683_c0":1.0})

rxn24352 = reaction_create('rxn24352','RXN-7657.c',0,1000,{"cpd00003_c0":-1.0,"cpd10408_c0":-1.0,"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd24683_c0":1.0})
rxn24352_reverse = reaction_create('rxn24352_reverse','RXN-7657.c',0,1000,{"cpd00003_c0":1.0,"cpd10408_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd24683_c0":-1.0})
ibutohtpp = reaction_create('ibutohtpp','ibutohtpp',0,1000,{"cpd10408_c0":-1.0,"cpd10408_e0":1.0})
EX_ibutohe_e = reaction_create('EX_ibutohe_e','EX_ibutohe_e',0,1000,{"cpd10408_e0":-1.0})

In [ ]:
etiZM564_iso = json_load(json_path)
rxnlist = [rxn24344,rxn24352,rxn24352_reverse,ibutohtpp,EX_ibutohe_e]
for i in rxnlist:
    if i not in model_json['reactions']:
        etiZM564_iso['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        etiZM564_iso['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_iso.json',"w") as f:
    json.dump(etiZM564_iso,f,indent=2)


cpd00067_c0
cpd00123_c0
cpd00011_c0
cpd00003_c0
cpd00004_c0


In [ ]:
model = cobra.io.load_json_model("./model/etiZM564_iso.json")
model.objective = "EX_ibutohe_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 100
o2_level = 0 # 0|1000
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
medium['EX_co2_e'] = 0
medium['EX_nh4_e'] = 1000
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 74

print(pfba_solution['EX_ibutohe_e'],pfba_solution['EX_ibutohe_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_ibutohe_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)

# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

100.0 1.0 0.4111111111111111


In [ ]:
model = read_json_model('./model/etiZM564_iso.json')
model.obj_name = "EX_ibutohe_e"
model.medium['EX_glc_D_e_reverse'] = 100
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 100)
o2_level = 0 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
model.get_reaction_by_id("EXT_gly_glu_e_reverse").bounds = (0, 0)
model.get_reaction_by_id("rxn08922_reverse").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 74

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

5.63768137871237 0.9999999999999597 0.41111111111109455


## 1,4-BDO(xylose)

In [ ]:
EX_xyl_D_e_reverse = reaction_create('EX_xyl_D_e_reverse','EX_xyl_D_e_reverse',0,1000,{"cpd00154_e0":1.0})
EXT_xyl_D_e = reaction_create('EXT_xyl_D_e','EX_xyl_D_e_reverse',0,1000,{"cpd00154_e0":-1.0,"cpd00154_c0":1.0})

rxn01041 =reaction_create('rxn01041','D-xylose:NAD+ 1-oxidoreductase',0,1000,{"cpd00003_c0":-1.0,"cpd00154_c0":-1.0,"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd01527_c0":1.0})
rxn01041_reverse =reaction_create('rxn01041_reverse','D-xylose:NAD+ 1-oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00154_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd01527_c0":-1.0})

rxn01751 =reaction_create('rxn01751','D-Xylonolactone lactonohydrolase',0,1000,{"cpd00001_c0":-1.0,"cpd01527_c0":-1.0,"cpd00067_c0":1.0,"cpd00391_c0":1.0})

rxn01753 =reaction_create('rxn01753','D-Xylonate hydro-lyase',0,1000,{"cpd00391_c0":-1.0,"cpd00001_c0":1.0,"cpd02386_c0":1.0})

rxn01306 =reaction_create('rxn01306','2-dehydro-3-deoxy-D-pentonate glycolaldehyde-lyase (pyruvate-forming)',0,1000,{"cpd02386_c0":-1.0,"cpd00020_c0":1.0,"cpd00229_c0":1.0})
rxn01306_reverse =reaction_create('rxn01306_reverse','2-dehydro-3-deoxy-D-pentonate glycolaldehyde-lyase (pyruvate-forming)',0,1000,{"cpd02386_c0":1.0,"cpd00020_c0":-1.0,"cpd00229_c0":-1.0})

# 2-dehydro-3-deoxy-D-pentonate -》 ketoglutarate semialdehyde
rxn01633=reaction_create('rxn01633','2-Dehydro-3-deoxy-L-arabinonate hydro-lyase',0,1000,{"cpd00520_c0":-1,"cpd00001_c0":1,"cpd00340_c0":1})
# ketoglutarate semialdehyde -》 2-Oxoglutarate
rxn00196=reaction_create('rxn00196','2,5-dioxopentanoate:NADP+ 5-oxidoreductase',0,1000,{"cpd00001_c0":-1,"cpd00006_c0":-1,"cpd00340_c0":-1,"cpd00005_c0":1,"cpd00024_c0":1,"cpd00067_c0":2})




# maea  
rxn00159_reverse = reaction_create('rxn00159_reverse','S_malate_NAD_plus__oxidoreductase_decarboxylating_c0',0,1000,{"cpd00003_c0":1.0,"cpd00130_c0":1.0,"cpd00004_c0":-1.0,"cpd00011_c0":-1.0,"cpd00020_c0":-1.0})
# mdh
rxn00248 = reaction_create('rxn00248','(S)-malate:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":-1.0,"cpd00130_c0":-1.0,"cpd00004_c0":1.0,"cpd00032_c0":1.0,"cpd00067_c0":1.0})
rxn00248_reverse = reaction_create('rxn00248_reverse','(S)-malate:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00130_c0":1.0,"cpd00004_c0":-1.0,"cpd00032_c0":-1.0,"cpd00067_c0":-1.0})
# frd
rxn00284 = reaction_create('rxn00284','succinate:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00036_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd00106_c0":-1.0})


# sucD
rxn28991 = reaction_create('rxn28991','Succinate thiokinase',0,1000,{"cpd00002_c0":-1.0,"cpd00010_c0":-1.0,"cpd00036_c0":-1.0,"cpd00008_c0":1.0,"cpd00009_c0":1.0,"cpd00078_c0":1.0})
rxn28991_reverse = reaction_create('rxn28991_reverse','Succinate thiokinase',0,1000,{"cpd00002_c0":1.0,"cpd00010_c0":1.0,"cpd00036_c0":1.0,"cpd00008_c0":-1.0,"cpd00009_c0":-1.0,"cpd00078_c0":-1.0})
# sucD_kel在文章中是BDO-succ
rxn16140 = reaction_create('rxn16140','succinate-semialdehyde:NADP+ oxidoreductase (CoA-acylating)',0,1000,{"cpd00005_c0":-1.0,"cpd00067_c0":-1.0,"cpd00078_c0":-1.0,"cpd00006_c0":1.0,"cpd00010_c0":1.0,"cpd00199_c0":1.0})
rxn16140_reverse = reaction_create('rxn16140_reverse','succinate-semialdehyde:NADP+ oxidoreductase (CoA-acylating)',0,1000,{"cpd00005_c0":1.0,"cpd00067_c0":1.0,"cpd00078_c0":1.0,"cpd00006_c0":-1.0,"cpd00010_c0":-1.0,"cpd00199_c0":-1.0})
# 4HBD
rxn38972 = reaction_create('rxn38972','RXN-6903.m.chlamyexp.NAD_NADH',0,1000,{"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd00199_c0":-1.0,"cpd00003_c0":1.0,"cpd00728_c0":1.0})
# Cat2
rxn03641 = reaction_create('rxn03641','acetyl-CoA: 4-hydroxybutanoate CoA transferase',0,1000,{"cpd00022_c0":-1.0,"cpd00728_c0":-1.0,"cpd00029_c0":1.0,"cpd07946_c0":1.0,"cpd00067_c0":1.0})
# Ald
rxn37165 = reaction_create('rxn37165','4-hydroxybutyryl-CoA reductase',0,1000,{"cpd00004_c0":-1.0,"cpd00067_c0":-2.0,"cpd07946_c0":-1.0,"cpd00003_c0":1.0,"cpd00010_c0":1.0,"cpd23932_c0":1.0})
rxn37165_reverse = reaction_create('rxn37165_reverse','4-hydroxybutyryl-CoA reductase',0,1000,{"cpd00004_c0":1.0,"cpd00067_c0":2.0,"cpd07946_c0":1.0,"cpd00003_c0":-1.0,"cpd00010_c0":-1.0,"cpd23932_c0":-1.0})
# Adh
rxn37167 = reaction_create('rxn37167','4-hydroxybutanal dehydrogenase',0,1000,{"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd23932_c0":-1.0,"cpd00003_c0":1.0,"cpd23934_c0":1.0})
rxn37167_reverse = reaction_create('rxn37167_reverse','4-hydroxybutanal dehydrogenase',0,1000,{"cpd00004_c0":1.0,"cpd00067_c0":1.0,"cpd23932_c0":1.0,"cpd00003_c0":-1.0,"cpd23934_c0":-1.0})

EXT_14BDO_e = reaction_create('EXT_14BDO_e','1,4 butylene glycol transporter',0,1000,{"cpd23934_c0":-1.0,"cpd23934_e0":1.0})
EX_14BDO_e = reaction_create('EX_14BDO_e','1,4 butylene glycol exchange',0,1000,{"cpd23934_e0":-1.0})

# menD-》BDO-akg
rxn00197 = reaction_create('rxn00197','2-oxoglutarate:NADP+ 2-oxidoreductase (CoA-succinylating)',0,1000,{"cpd00006_c0":-1.0,"cpd00010_c0":-1.0,"cpd00024_c0":-1.0,"cpd00005_c0":1.0,"cpd00011_c0":1.0,"cpd00078_c0":1.0})
rxn00197_NAD = reaction_create('rxn00197_NAD','2-oxoglutarate:NAD+ 2-oxidoreductase (CoA-succinylating)',0,1000,{"cpd00003_c0":-1.0,"cpd00010_c0":-1.0,"cpd00024_c0":-1.0,"cpd00004_c0":1.0,"cpd00011_c0":1.0,"cpd00078_c0":1.0})

In [ ]:
# pck
rxn35519 = reaction_create('rxn35519','phosphoenolpyruvate carboxykinase, mitochondria',0,1000,{"cpd00002_c0":-1.0,"cpd00032_c0":-1.0,"cpd00008_c0":1.0,"cpd00011_c0":1.0,"cpd00061_c0":1.0})
rxn35519_reverse = reaction_create('rxn35519_reverse','phosphoenolpyruvate carboxykinase, mitochondria',0,1000,{"cpd00002_c0":1.0,"cpd00032_c0":1.0,"cpd00008_c0":-1.0,"cpd00011_c0":-1.0,"cpd00061_c0":-1.0})


In [ ]:
rxnlist = [rxn01633,rxn00196,rxn35519,rxn35519_reverse,rxn01306,rxn01306_reverse,rxn01751,rxn01753,EX_xyl_D_e_reverse,EXT_xyl_D_e,rxn01041_reverse,rxn01041,rxn00159_reverse,rxn00248,rxn00248_reverse,rxn00284,rxn00197,rxn00197_NAD,rxn28991,rxn28991_reverse,rxn16140,rxn16140_reverse,rxn38972,rxn03641,rxn37165,rxn37165_reverse,rxn37167,rxn37167_reverse,EXT_14BDO_e,EX_14BDO_e]

etiZM564_14BDO_xyl = json_load(json_path)
for i in rxnlist:
    if i not in model_json['reactions']:
        etiZM564_14BDO_xyl['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        etiZM564_14BDO_xyl['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_14BDO_xyl.json',"w") as f:
    json.dump(etiZM564_14BDO_xyl,f,indent=2)


cpd00001_c0
cpd00340_c0
cpd00006_c0
cpd00005_c0
cpd00024_c0
cpd00067_c0
cpd00002_c0
cpd00032_c0
cpd00008_c0
cpd00011_c0
cpd00061_c0
cpd00020_c0
cpd00229_c0
cpd00154_e0
cpd00154_c0
cpd00003_c0
cpd00004_c0
cpd00130_c0
cpd00036_c0
cpd00106_c0
cpd00010_c0
cpd00078_c0
cpd00009_c0
cpd00199_c0
cpd00022_c0
cpd00029_c0


In [ ]:
model = cobra.io.load_json_model('./model/etiZM564_14BDO_xyl.json')
model.objective = "EX_14BDO_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 0
model.reactions.get_by_id("EX_glc_D_e_reverse").bounds = (0,0)
medium['EX_xyl_D_e_reverse'] = 100
model.reactions.get_by_id("EX_xyl_D_e_reverse").bounds = (0,100)
o2_level = 0 # 0|1000
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0,o2_level)
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)
pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 90

print(pfba_solution['EX_14BDO_e'],pfba_solution['EX_14BDO_e']/pfba_solution['EX_xyl_D_e_reverse'],pfba_solution['EX_14BDO_e']/pfba_solution['EX_xyl_D_e_reverse']*product_mass/150)
dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
for i in dict_fluxes.keys():
    print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

0.0 nan nan


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11872\1259097143.py:16: RuntimeWarning: invalid value encountered in double_scalars
  print(pfba_solution['EX_14BDO_e'],pfba_solution['EX_14BDO_e']/pfba_solution['EX_xyl_D_e_reverse'],pfba_solution['EX_14BDO_e']/pfba_solution['EX_xyl_D_e_reverse']*product_mass/150)


In [ ]:
model = read_json_model('./model/etiZM564_14BDO_xyl.json')
model.obj_name = "EX_14BDO_e"
model.medium['EX_glc_D_e_reverse'] = 0
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 0)
model.medium['EX_xyl_D_e_reverse'] = 100
model.get_reaction_by_id("EX_xyl_D_e_reverse").bounds = (0, 100)
o2_level = 1000 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 90

print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_xyl_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_xyl_D_e_reverse"].value*product_mass/150)
for i in model.reactions:
    if opt_etm.reaction[i].value > 1e-4:
        print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

5.576124132787161 0.3809523809523946 0.22857142857143675
EX_ac_e	Acetate <=> 	(0.0, 1000.0)	5.576
EX_co2_e	CO2 <=> 	(0.0, 1000.0)	10.455
EX_gcald_e	Glycolaldehyde <=> 	(0.0, 1000.0)	14.637
EX_h_e	H+ <=> 	(0.0, 1000.0)	5.576
EX_h2o_e	H2O <=> 	(0.0, 1000.0)	4.879
rxn00154	NAD + CoA + Pyruvate <=> NADH + CO2 + Acetyl-CoA	(0.0, 1000.0)	9.061
rxn00198	NADP + Isocitrate <=> NADPH + CO2 + 2-Oxoglutarate	(0.0, 1000.0)	3.485
rxn00256_num1	H2O + Acetyl-CoA + Oxaloacetate <=> CoA + H+ + Citrate	(0.0, 1000.0)	3.485
rxn00799	L-Malate <=> H2O + Fumarate	(0.0, 1000.0)	2.091
rxn00974	Citrate <=> H2O + cis-Aconitate	(0.0, 1000.0)	3.485
rxn08173	ADP + Phosphate + H+ <=> H2O + ATP + H+	(0.0, 1000.0)	2.091
rxn08237	CO2 <=> CO2	(0.0, 1000.0)	10.455
rxn09032	O2 <=> O2	(0.0, 1000.0)	2.091
rxn09385	Xylose <=> Xylose	(0.0, 1000.0)	14.637
rxn09752	Glycolaldehyde <=> Glycolaldehyde	(0.0, 1000.0)	14.637
rxn36773	NADH + H+ + Ubidecarenone <=> NAD + ubiquinol(10)	(0.0, 1000.0)	4.182
rxn37453	O2 + H+ + ubiquinol(10)

## Succinate pathway

In [ ]:
rxn00248 = reaction_create('rxn00248','(S)-malate:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":-1.0,"cpd00130_c0":-1.0,"cpd00004_c0":1.0,"cpd00032_c0":1.0,"cpd00067_c0":1.0})
rxn00248_reverse = reaction_create('rxn00248_reverse','(S)-malate:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00130_c0":1.0,"cpd00004_c0":-1.0,"cpd00032_c0":-1.0,"cpd00067_c0":-1.0})

# maea  
rxn00159_reverse = reaction_create('rxn00159_reverse','S_malate_NAD_plus__oxidoreductase_decarboxylating_c0',0,1000,{"cpd00003_c0":1.0,"cpd00130_c0":1.0,"cpd00004_c0":-1.0,"cpd00011_c0":-1.0,"cpd00020_c0":-1.0})

rxn00284 = reaction_create('rxn00284','succinate:NAD+ oxidoreductase',0,1000,{"cpd00003_c0":1.0,"cpd00036_c0":1.0,"cpd00004_c0":-1.0,"cpd00067_c0":-1.0,"cpd00106_c0":-1.0})

rxn08094 = reaction_create('rxn08094','2-Oxoglutarate dehydrogenase complex',0,1000,{"cpd00003_c0":-1.0,"cpd00010_c0":-1.0,"cpd00024_c0":-1.0,"cpd00004_c0":1.0,"cpd00011_c0":1.0,"cpd00078_c0":1.0})

In [ ]:
rxnlist = [rxn00248,rxn00248_reverse,rxn00159_reverse,rxn00284,rxn08094]
model_iZM564_succ1 = json_load(json_path)
for i in rxnlist:
    if i not in model_json['reactions']:
        model_iZM564_succ1['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        model_iZM564_succ1['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_succ1.json',"w") as f:
    json.dump(model_iZM564_succ1,f,indent=2)


cpd00003_c0
cpd00130_c0
cpd00004_c0
cpd00032_c0
cpd00067_c0
cpd00011_c0
cpd00020_c0
cpd00036_c0
cpd00106_c0
cpd00010_c0
cpd00024_c0
cpd00078_c0


In [ ]:
model = cobra.io.load_json_model("./model/etiZM564_succ1.json")
model.objective = "EX_succ_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 40
# 三挡，无氧，有氧，有氧+CO2
o2_level = 0 # 0|1000
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
medium['EX_co2_e'] = o2_level
medium['EX_nh4_e'] = 1000
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)

# model.reactions.get_by_id("rxn05937").bounds = (0,0)

pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 116

print(pfba_solution['EX_succ_e'],pfba_solution['EX_succ_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_succ_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)

dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
for i in dict_fluxes.keys():
    print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

56.84210526315788 1.4210526315789471 0.9157894736842104
EX_etoh_e	Ethanol --> 	(0, 1000.0)	6.316
EX_h_e	H+ --> 	(0, 1000.0)	145.263
EX_h2o_e	H2O --> 	(0, 1000.0)	31.579
EX_h2s_e	H2S --> 	(0, 1000.0)	6.316
EX_succ_e	Succinate --> 	(0, 1000.0)	56.842
rxn00137	H2O + Adenosine 3-5-bisphosphate --> Phosphate + AMP	(0.0, 1000.0)	6.316
rxn00148	ATP + Pyruvate <-- ADP + Phosphoenolpyruvate + H+	(-1000.0, 0.0)	-40.000
rxn00154	NAD + CoA + Pyruvate --> NADH + CO2 + Acetyl-CoA	(0.0, 1000.0)	16.842
rxn00168	Pyruvate + H+ --> CO2 + Acetaldehyde	(0.0, 1000.0)	6.316
rxn00198	NADP + Isocitrate --> NADPH + CO2 + 2-Oxoglutarate	(0.0, 1000.0)	4.211
rxn00216	ATP + D-Glucose --> ADP + H+ + D-glucose-6-phosphate	(0.0, 1000.0)	40.000
rxn00361	ATP + APS --> ADP + 3-phosphoadenylylsulfate + H+	(0.0, 1000.0)	6.316
rxn00379_num1	ATP + Sulfate --> PPi + APS	(0.0, 1000.0)	6.316
rxn00459	2-Phospho-D-glycerate --> H2O + Phosphoenolpyruvate	(0.0, 1000.0)	40.000
rxn00505	NAD + Isocitrate --> NADH + CO2 + 2-Oxoglutarat

In [ ]:
model = read_json_model('./model/etiZM564_succ1.json')
model.obj_name = "EX_succ_e"
model.medium['EX_glc_D_e_reverse'] = 1000
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 1000)
o2_level = 1000 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.medium['EX_co2_e'] = 1000
model.get_reaction_by_id("EX_co2_e").bounds = (-1000, 1000)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)

product_mass = 116
print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

88.69291673834272 1.1999999999998432 0.7733333333332322


# C5 Product
## 衣康酸 (Itaconate)

In [ ]:
rxn01605 = reaction_create('rxn01605','cis-Aconitate carboxy-lyase',0,1000,{"cpd00067_c0":-1.0,"cpd00331_c0":-1.0,"cpd00011_c0":1.0,"cpd00380_c0":1.0})

EX_Itaconate_e = reaction_create('EX_Itaconate_e','Itaconate exchange',0,1000,{"cpd00380_e0":-1.0})

rxn12562 = reaction_create('rxn12562','Itaconate transport via proton symport',0,1000,{"cpd00067_e0":2.0,"cpd00380_e0":1.0,"cpd00067_c0":-2.0,"cpd00380_c0":-1.0})

In [ ]:
model_iZM564_Itaconate = json_load(json_path)
rxnlist = [rxn01605,EX_Itaconate_e,rxn12562]
for i in rxnlist:
    if i not in model_json['reactions']:
        model_iZM564_Itaconate['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        model_iZM564_Itaconate['metabolites'].append(metdict[newmet])


cpd00067_c0
cpd00331_c0
cpd00011_c0
cpd00067_e0


In [ ]:
with open('./model/etiZM564_Itaconate.json',"w") as f:
    json.dump(model_iZM564_Itaconate,f,indent=2)

In [ ]:
model = read_json_model('./model/etiZM564_Itaconate.json')
model.obj_name = "EX_Itaconate_e"
model.medium['EX_glc_D_e_reverse'] = 35
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 35)
o2_level = 1000 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
opt_etm = calculate_ETM(model, "maximize", 0.0)

product_mass = 128
print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_glc_D_e_reverse"].value*product_mass/180)
# for i in model.reactions:
#     if opt_etm.reaction[i].value > 1e-4:
#         print(f"{i}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

20.557044036880495 0.394366197183071 0.28043818466351716


In [ ]:
model = cobra.io.load_json_model('./model/etiZM564_Itaconate.json')
model.objective = "EX_Itaconate_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 100
o2_level = 1000 # 0|1000
medium['EX_o2_e'] = 0
model.reactions.get_by_id("EX_o2_e").bounds = (0, 0)
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.medium = medium
print(model.medium)
model.reactions.get_by_id("ATPM").bounds = (0,0)

pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 128

print(pfba_solution['EX_Itaconate_e'],pfba_solution['EX_Itaconate_e']/pfba_solution['EX_glc_D_e_reverse'],pfba_solution['EX_Itaconate_e']/pfba_solution['EX_glc_D_e_reverse']*product_mass/180)

# dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
# for i in dict_fluxes.keys():
#     print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

{'EX_ca2_e_reverse': 1000.0, 'EX_cd2_e_reverse': 1000.0, 'EX_cl_e_reverse': 1000.0, 'EX_cobalt2_e_reverse': 1000.0, 'EX_cu2_e_reverse': 1000.0, 'EX_fe2_e_reverse': 1000.0, 'EX_fe3_e_reverse': 1000.0, 'EX_glc_D_e_reverse': 100, 'EX_h_e_reverse': 1000.0, 'EX_h2o_e_reverse': 1000.0, 'EX_hg2_e_reverse': 1000.0, 'EX_k_e_reverse': 1000.0, 'EX_mg2_e_reverse': 1000.0, 'EX_mn2_e_reverse': 1000.0, 'EX_mobd_e_reverse': 1000.0, 'EX_na1_e_reverse': 1000.0, 'EX_nh4_e_reverse': 1000.0, 'EX_Pb_e_reverse': 1000.0, 'EX_pi_e_reverse': 1000.0, 'EX_so4_e_reverse': 1000.0, 'EX_thm_e_reverse': 1000.0, 'EX_zn2_e_reverse': 1000.0, 'EX_pnto_R_e_reverse': 1000.0, 'EX_btn_e_reverse': 1000.0, 'EX_o2_e_reverse': 1000}
100.0 1.0 0.7111111111111111


# Xylitol 木糖酸

In [ ]:
EX_xyl_D_e_reverse = reaction_create('EX_xyl_D_e_reverse','EX_xyl_D_e_reverse',0,1000,{"cpd00154_e0":1.0})
rxn09384 = reaction_create('rxn09384','D-xylose transport in via proton symport (periplasm)',0,1000,{"cpd00067_p0":-1.0,"cpd00154_p0":-1.0,"cpd00067_c0":1.0,"cpd00154_c0":1.0})
# rxn09385 = reaction_create('rxn09385','D-xylose transport via diffusion (extracellular to periplasm)',0,1000,{"cpd00154_p0":1.0,"cpd00154_e0":-1.0})


EXT_xylitol_e = reaction_create('EXT_xylitol_e','xylitol EXT',0,1000,{"cpd00306_c0":-1.0,"cpd00306_e0":1.0})
EX_xylitol_e = reaction_create('EX_xylitol_e','xylitol Exchange',0,1000,{"cpd00306_e0":-1.0})

In [ ]:
rxnlist = [EX_xyl_D_e_reverse,rxn09384,EXT_xylitol_e,EX_xylitol_e]
etiZM564_xylitol = json_load(json_path)
for i in rxnlist:
    if i not in model_json['reactions']:
        etiZM564_xylitol['reactions'].append(dict(i))
metdict = process_metabolites(rxnlist, bigmodel)
            
for newmet in metdict.keys():
    try:
        print(model_iZM564.metabolites.get_by_id(newmet))
    except:
        etiZM564_xylitol['metabolites'].append(metdict[newmet])

with open('./model/etiZM564_xylitol.json',"w") as f:
    json.dump(etiZM564_xylitol,f,indent=2)


cpd00154_e0
cpd00067_p0
cpd00067_c0
cpd00154_c0
cpd00306_c0


In [ ]:
model = cobra.io.load_json_model('./model/etiZM564_xylitol.json')
model.objective = "EX_xylitol_e" # rxn10952 EX_succ_e EX_etoh_e EX_lac_D_e
medium = model.medium
medium['EX_glc_D_e_reverse'] = 100
model.reactions.get_by_id("EX_glc_D_e_reverse").bounds = (0,100)
medium['EX_xyl_D_e_reverse'] = 100
model.reactions.get_by_id("EX_xyl_D_e_reverse").bounds = (0,100)
o2_level = 0 # 0|1000
medium['EX_o2_e_reverse'] = o2_level
model.reactions.get_by_id("EX_o2_e_reverse").bounds = (0,o2_level)
model.medium = medium
model.reactions.get_by_id("ATPM").bounds = (0,0)

pfba_solution = cobra.flux_analysis.pfba(model)
product_mass = 152

print(pfba_solution['EX_xylitol_e'],pfba_solution['EX_xylitol_e']/pfba_solution['EX_xyl_D_e_reverse'],pfba_solution['EX_xylitol_e']/pfba_solution['EX_xyl_D_e_reverse']*product_mass/150)
dict_fluxes = dict(pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4])
for i in dict_fluxes.keys():
    print(f"{i}\t{model.reactions.get_by_id(i).build_reaction_string(use_metabolite_names=True)}\t{model.reactions.get_by_id(i).bounds}\t{dict_fluxes[i]:.3f}")

100.0 1.0 1.0133333333333334
EX_ac_e	Acetate --> 	(0, 1000.0)	16.667
EX_co2_e	CO2 --> 	(0, 1000.0)	33.333
EX_h_e	H+ --> 	(0, 1000.0)	50.000
rxn00148	ATP + Pyruvate <-- ADP + Phosphoenolpyruvate + H+	(-1000.0, 0.0)	-8.333
rxn00154	NAD + CoA + Pyruvate --> NADH + CO2 + Acetyl-CoA	(0.0, 1000.0)	16.667
rxn00168	Pyruvate + H+ --> CO2 + Acetaldehyde	(0.0, 1000.0)	16.667
rxn00198	NADP + Isocitrate --> NADPH + CO2 + 2-Oxoglutarate	(0.0, 1000.0)	16.667
rxn00216	ATP + D-Glucose --> ADP + H+ + D-glucose-6-phosphate	(0.0, 1000.0)	25.000
rxn00251	Phosphate + Oxaloacetate + H+ <-- H2O + CO2 + Phosphoenolpyruvate	(-1000.0, 0.0)	-16.667
rxn00459	2-Phospho-D-glycerate --> H2O + Phosphoenolpyruvate	(0.0, 1000.0)	25.000
rxn00507	H2O + NADP + Acetaldehyde --> NADPH + Acetate + 2.0 H+	(0.0, 1000.0)	16.667
rxn00604	NADP + D-glucose-6-phosphate --> NADPH + H+ + 6-phospho-D-glucono-1-5-lactone	(0.0, 1000.0)	25.000
rxn00612	NADP + Glycerol-3-phosphate --> NADPH + H+ + Glycerone-phosphate	(0.0, 1000.0)	41.667
r

In [ ]:
model = read_json_model('./model/etiZM564_xylitol.json')
model.obj_name = "EX_xylitol_e"
model.medium['EX_glc_D_e_reverse'] = 100
model.get_reaction_by_id("EX_glc_D_e_reverse").bounds = (0, 100)
model.medium['EX_xyl_D_e_reverse'] = 100
model.get_reaction_by_id("EX_xyl_D_e_reverse").bounds = (0, 100)
o2_level = 0 # 0|1000
model.medium['EX_o2_e_reverse'] = o2_level
model.get_reaction_by_id("EX_o2_e_reverse").bounds = (0, o2_level)
model.get_reaction_by_id("ATPM").bounds = (0, 0)
model.reactions["rxn01043"].g0 = 0

opt_etm = calculate_ETM(model, "maximize", 0.0)
product_mass = 152
print(opt_etm.obj(),opt_etm.obj()/opt_etm.reaction["EX_xyl_D_e_reverse"].value,opt_etm.obj()/opt_etm.reaction["EX_xyl_D_e_reverse"].value*product_mass/150)
for i in model.reactions:
    if opt_etm.reaction[i].value > 1e-4:
        print(f"{i}\t{model.reactions[i].g0}\t{model.reactions[i].build_reaction_string(model,use_metabolite_names=True)}\t{model.reactions[i].bounds}\t{opt_etm.reaction[i].value:.3f}")

100.0 1.0 1.0133333333333334
ACTNtex	0.0	(S)-Acetoin <=> (S)-Acetoin	(0.0, 1000.0)	9.994
ACTNtpp	0.0	(S)-Acetoin <=> (S)-Acetoin	(0.0, 1000.0)	9.994
EX_ac_e	0.0	Acetate <=> 	(0.0, 1000.0)	0.009
EX_actn__S_e	0.0	(S)-Acetoin <=> 	(0.0, 1000.0)	9.994
EX_co2_e	0.0	CO2 <=> 	(0.0, 1000.0)	39.996
EX_h_e	0.0	H+ <=> 	(0.0, 1000.0)	40.006
rxn00148	0.0	ATP + Pyruvate <=> ADP + Phosphoenolpyruvate + H+	(-1000.0, 0.0)	9.999
rxn00154	-35.2	NAD + CoA + Pyruvate <=> NADH + CO2 + Acetyl-CoA	(0.0, 1000.0)	19.998
rxn00198	6.6	NADP + Isocitrate <=> NADPH + CO2 + 2-Oxoglutarate	(0.0, 1000.0)	19.998
rxn00216	-13.3	ATP + D-Glucose <=> ADP + H+ + D-glucose-6-phosphate	(0.0, 1000.0)	29.997
rxn00251	33.5	Phosphate + Oxaloacetate + H+ <=> H2O + CO2 + Phosphoenolpyruvate	(-1000.0, 0.0)	19.998
rxn00459	-4.1	2-Phospho-D-glycerate <=> H2O + Phosphoenolpyruvate	(0.0, 1000.0)	29.997
rxn00507	-34.5	H2O + NADP + Acetaldehyde <=> NADPH + Acetate + H+	(0.0, 1000.0)	0.009
rxn00604	3.3	NADP + D-glucose-6-phosphate <=> NADPH